In [1]:
from pathlib import Path
import zipfile, io, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option('display.max_colwidth', 130)
pd.set_option('display.max_columns', 40)
ROOT=Path('data')
ZIP=Path('data.zip')
if not ROOT.exists() and not ZIP.exists():
    raise FileNotFoundError('Place data.zip ou le dossier data/ dans le répertoire courant.')
def load(area, filename):

    if area == "ecology":
        folder = ROOT / "ecology" / "ecology-clean"

    elif area == "health":
        folder = ROOT / "health" / "health_clean"

    elif area == "logistics":
        folder = ROOT / "logistics" / "clean"

    else:
        folder = ROOT / area

    # Chercher le fichier avec ou sans suffixe _clean
    paths = [
        folder / f"{filename}_clean.csv",
        folder / f"{filename}.csv",
        ROOT / area / f"{filename}.csv"
    ]

    for path in paths:
        if path.exists():
            print(f"Fichier chargé : {path}")
            df = pd.read_csv(path, low_memory=False)
            df.columns = df.columns.str.strip()
            return df

    raise FileNotFoundError(
        f"Fichier introuvable : {filename}\n"
        f"Chemins testés : {[str(p) for p in paths]}"
    )
D={}
for area,names in {'ecology':['bat_surveys','environmental_field_reports','environmental_monthly','device_operations','monitoring_sites','orchard_surveys','orchard_site_profiles','panel_vergers_appareils'], 'health':['cardiac_recordings','clinical_encounters','clinical_notes','clinical_observations','clinical_procedures','patient_administration'], 'logistics':['erp_business_partners','erp_facilities','goods_receipts','inventory_adjustments','inventory_month_end','purchase_order_lines','shipments'], 'public_records':['institutional_memos','licensed_facilities','licensed_professionals','officer_filings','public_and_investigative_documents','public_appointments','registered_entities','regulatory_case_events','site_access_case_events']}.items():
    for name in names: D[name]=load(area,name)
print('Fichiers chargés :',len(D))
display(pd.DataFrame([(k,len(v),len(v.columns)) for k,v in D.items()],columns=['fichier','lignes','colonnes']))
def norm(s): return s.astype('string').str.strip().str.lower().str.replace(r'\s+',' ',regex=True)
def date(s):return pd.to_datetime(s,errors='coerce',format='mixed')
def show(df,cols=None,n=20):display(df[cols].head(n) if cols else df.head(n))
def evidence(label, df, cols=None, n=15):
    print(f'\n{label} : {len(df)} lignes')
    show(df,cols,n)


Fichier chargé : data\ecology\ecology-clean\bat_surveys_clean.csv
Fichier chargé : data\ecology\ecology-clean\environmental_field_reports_clean.csv
Fichier chargé : data\ecology\ecology-clean\environmental_monthly_clean.csv
Fichier chargé : data\ecology\ecology-clean\device_operations_clean.csv
Fichier chargé : data\ecology\ecology-clean\monitoring_sites_clean.csv
Fichier chargé : data\ecology\ecology-clean\orchard_surveys_clean.csv
Fichier chargé : data\ecology\ecology-clean\orchard_site_profiles_clean.csv
Fichier chargé : data\ecology\ecology-clean\panel_vergers_appareils_clean.csv
Fichier chargé : data\health\health_clean\cardiac_recordings.csv
Fichier chargé : data\health\health_clean\clinical_encounters.csv
Fichier chargé : data\health\health_clean\clinical_notes.csv
Fichier chargé : data\health\health_clean\clinical_observations.csv
Fichier chargé : data\health\health_clean\clinical_procedures.csv
Fichier chargé : data\health\health_clean\patient_administration.csv
Fichier chargé

,fichier,lignes,colonnes
0,bat_surveys,4992,14
1,environmental_field_reports,30,12
2,environmental_monthly,4992,11
3,device_operations,4992,19
4,monitoring_sites,64,20
5,orchard_surveys,1248,20
6,orchard_site_profiles,16,18
7,panel_vergers_appareils,1248,15
8,cardiac_recordings,44,19
9,clinical_encounters,1067,12


# Lien entre ecology et logistics 

## Analyse des concordances spatio-temporelles entre les accès aux sites écologiques et les expéditions biologiques
Cette analyse vise à rechercher un lien potentiel entre les interventions réalisées sur les sites écologiques et les expéditions de matériel biologique. Pour cela, nous avons croisé les données des accès aux sites (site_access_case_events), des sites de surveillance (monitoring_sites), des expéditions (shipments) et des établissements logistiques (erp_facilities). Nous avons d'abord sélectionné les événements correspondant à la fin d'un accès ainsi que les expéditions d'échantillons biologiques. Nous avons ensuite effectué une jointure selon leur région géographique et calculé le délai entre la fin de chaque intervention et le départ des marchandises. Seules les expéditions effectuées dans les 48 heures suivant un accès ont été conservées, afin d'identifier d'éventuelles concordances géographiques et temporelles

In [2]:
import pandas as pd

sites = load("ecology", "monitoring_sites")
access = load("public_records", "site_access_case_events")
shipments = load("logistics", "shipments")
facilities = load("logistics", "erp_facilities")

# 1. Normaliser les dates
access["event_time"] = pd.to_datetime(
    access["event_time"], errors="coerce"
)

shipments["dispatch_time"] = pd.to_datetime(
    shipments["dispatch_time"], errors="coerce"
)

# 2. Conserver les fins d'accès aux sites
access_end = access[
    access["activity"].astype(str).str.strip().str.lower()
    == "access_completed"
].copy()

# 3. Associer chaque accès à la région du site
access_end = access_end.merge(
    sites[["site_code", "site_name", "region"]],
    on="site_code",
    how="left",
    validate="many_to_one"
)

# 4. Conserver les expéditions biologiques
bio = shipments[
    shipments["item_category"].astype(str).str.strip().str.lower()
    == "biological_sample"
].copy()

# 5. Ajouter la région de l'établissement de départ
bio = bio.merge(
    facilities[["facility_code", "facility_name", "region"]],
    left_on="origin_facility_code",
    right_on="facility_code",
    how="left",
    validate="many_to_one",
    suffixes=("", "_origin")
)

# 6. Jointure par région
matches = access_end.merge(
    bio,
    on="region",
    how="inner",
    suffixes=("_access", "_shipment")
)

# 7. Calculer le délai entre la fin de l'accès et l'expédition
matches["delay_hours"] = (
    matches["dispatch_time"] - matches["event_time"]
).dt.total_seconds() / 3600

# 8. Garder les expéditions dans les 48 h suivantes
matches = matches[
    matches["delay_hours"].between(0, 48)
].copy()

# 9. Afficher les correspondances
result = matches[[
    "site_code",
    "site_name",
    "event_time",
    "shipment_number",
    "dispatch_time",
    "delay_hours",
    "facility_name",
    "destination_facility_code",
    "declared_quantity",
    "declared_purpose"
]].sort_values("delay_hours")

print("Correspondances trouvées :", len(result))
print("LIEN ENTRE ACCÈS AUX SITES ET EXPÉDITIONS")
print("Correspondances trouvées :", len(result))
print("Sites concernés :", result["site_code"].nunique())
print("Expéditions distinctes :", result["shipment_number"].nunique())

display(
    result[[
        "site_code",
        "event_time",
        "shipment_number",
        "dispatch_time",
        "delay_hours"
    ]].head(10)
)

Fichier chargé : data\ecology\ecology-clean\monitoring_sites_clean.csv
Fichier chargé : data\public_records\site_access_case_events.csv
Fichier chargé : data\logistics\shipments.csv
Fichier chargé : data\logistics\clean\erp_facilities_clean.csv
Correspondances trouvées : 19
LIEN ENTRE ACCÈS AUX SITES ET EXPÉDITIONS
Correspondances trouvées : 19
Sites concernés : 9
Expéditions distinctes : 18


,site_code,event_time,shipment_number,dispatch_time,delay_hours
1,TMB-NAC,2018-02-16 03:00:00,SHP-2018-00004,2018-02-16 09:00:00,6.0
11,KNT-KES,2018-03-23 03:00:00,SHP-2018-00007,2018-03-23 09:00:00,6.0
20,AVL-MAR,2018-05-25 03:00:00,SHP-2018-00032,2018-05-25 09:00:00,6.0
31,TMB-PEL,2019-08-09 03:00:00,SHP-2019-00009,2019-08-09 09:00:00,6.0
42,KNT-LUM,2019-09-13 03:00:00,SHP-2019-00026,2019-09-13 09:00:00,6.0
50,AVL-VEL,2019-11-08 03:00:00,SHP-2019-00041,2019-11-08 09:00:00,6.0
62,KNT-SEN,2020-04-10 03:00:00,SHP-2020-00019,2020-04-10 09:00:00,6.0
70,AVL-ORR,2020-06-12 03:00:00,SHP-2020-00038,2020-06-12 09:00:00,6.0
82,KNT-KES,2021-03-26 03:00:00,SHP-2021-00007,2021-03-26 09:00:00,6.0
90,AVL-MAR,2021-05-21 03:00:00,SHP-2021-00028,2021-05-21 09:00:00,6.0


La répétition d'un délai exactement égal à six heures entre la fin des interventions écologiques et le départ des expéditions biologiques suggère une possible coordination entre ces deux activités, renforçant l'hypothèse d'un circuit organisé de collecte et de transport de ressources biologiques.

## Analyse de l'évolution de l'activité des chauves-souris avant et après les expéditions biologiques
cette analyse vise à déterminer si les interventions écologiques associées aux expéditions biologiques sont suivies d'une diminution de l'activité des chauves-souris. Pour cela, nous avons comparé le taux d'observation (nombre de chauves-souris observées par heure) sur les trois mois précédant et suivant chaque intervention.

In [16]:
bats = D["bat_surveys"].copy()

# Récupérer les événements identifiés précédemment
events = matches[
    ["site_code", "event_time", "shipment_number"]
].drop_duplicates().copy()

# Uniformiser les codes des sites
bats["site_code"] = bats["site_code"].astype("string").str.strip().str.upper()
events["site_code"] = events["site_code"].astype("string").str.strip().str.upper()

# Convertir les dates et les valeurs numériques
bats["survey_start"] = pd.to_datetime(
    bats["survey_start"], format="mixed", errors="coerce"
)

events["event_time"] = pd.to_datetime(
    events["event_time"], format="mixed", errors="coerce"
)

for col in ["observed_bat_count", "observation_hours"]:
    bats[col] = pd.to_numeric(bats[col], errors="coerce")

# Une seule analyse par événement d'accès sur un site
events = events.drop_duplicates(
    subset=["site_code", "event_time"]
)

results = []

for _, event in events.iterrows():

    date = event["event_time"]

    if pd.isna(date):
        continue

    site_data = bats[
        (bats["site_code"] == event["site_code"])
        & bats["survey_start"].notna()
        & bats["observed_bat_count"].notna()
        & (bats["observation_hours"] > 0)
    ]

    # Relevés des 3 mois précédents
    before = site_data[
        (site_data["survey_start"] < date)
        & (site_data["survey_start"] >= date - pd.DateOffset(months=3))
    ]

    # Relevés des 3 mois suivants
    after = site_data[
        (site_data["survey_start"] > date)
        & (site_data["survey_start"] <= date + pd.DateOffset(months=3))
    ]

    # Au moins 2 relevés avant et après
    if len(before) < 2 or len(after) < 2:
        continue

    # Taux d'observation
    rate_before = (
        before["observed_bat_count"].sum()
        / before["observation_hours"].sum()
    )

    rate_after = (
        after["observed_bat_count"].sum()
        / after["observation_hours"].sum()
    )

    # Variation en pourcentage
    change = (
        (rate_after - rate_before) / rate_before * 100
        if rate_before > 0 else np.nan
    )

    results.append({
        "site_code": event["site_code"],
        "event_date": date,
        "rate_before": rate_before,
        "rate_after": rate_after,
        "change_pct": change,
        "n_before": len(before),
        "n_after": len(after)
    })

comparison = pd.DataFrame(results)

print("=== ÉVOLUTION DES OBSERVATIONS DE CHAUVES-SOURIS ===")
print("Événements identifiés :", len(events))
print("Événements analysables :", len(comparison))

if not comparison.empty:

    comparison = comparison.sort_values("change_pct")

    print(
        "Événements suivis d'une baisse :",
        (comparison["change_pct"] < 0).sum()
    )

    print(
        "Variation médiane :",
        f"{comparison['change_pct'].median():.2f}%"
    )

    display(
        comparison[
            ["site_code", "rate_before", "rate_after", "change_pct"]
        ].head(10).round(3)
    )

else:
    print("Pas assez de relevés pour effectuer la comparaison.")

=== ÉVOLUTION DES OBSERVATIONS DE CHAUVES-SOURIS ===
Événements identifiés : 19
Événements analysables : 15
Événements suivis d'une baisse : 15
Variation médiane : -45.79%


,site_code,rate_before,rate_after,change_pct
7,TMB-NAC,0.624,0.145,-76.693
6,AVL-MAR,0.779,0.230,-70.517
1,KNT-LUM,0.402,0.125,-68.955
5,KNT-KES,0.784,0.252,-67.933
12,TMB-PEL,0.378,0.121,-67.885
9,AVL-VEL,0.566,0.236,-58.234
2,AVL-VEL,0.579,0.298,-48.493
4,AVL-ORR,0.505,0.274,-45.789
10,KNT-KES,0.571,0.315,-44.801
0,TMB-PEL,0.279,0.156,-43.839


Sur les 19 correspondances identifiées entre interventions et expéditions biologiques, 15 disposent de suffisamment de relevés pour être analysées. Les 4 autres sont exclues en raison de l'absence de données suffisantes avant ou après l'intervention.

Les résultats montrent que les 15 événements analysés sont suivis d'une baisse du taux d'observation des chauves-souris, avec des diminutions comprises entre 37,77 % et 76,69 %.

Plusieurs cas se distinguent :

TMB-NAC (2022) : baisse de 76,69 %.

AVL-MAR (2021) : baisse de 70,52 %.

KNT-LUM (2019) : baisse de 68,95 %.

KNT-KES (2021) : baisse de 67,93 %.

Le site KNT-KES est particulièrement intéressant, car une diminution est observée après deux interventions distinctes : −67,93 % en 2021 et −44,80 % en 2024.

## Comparaison avec les sites témoins

Afin de vérifier si les baisses observées sont spécifiques aux sites concernés par les interventions, nous avons comparé leur évolution à celle d'autres sites de la même région, sans correspondance intervention-expédition identifiée, sur les mêmes périodes de trois mois.

In [4]:
# Sites concernés par les interventions/expéditions
sites_suspects = set(events["site_code"].dropna())

def taux_observation(df):
    heures = df["observation_hours"].sum()
    return df["observed_bat_count"].sum() / heures if heures > 0 else np.nan

controles = []

for _, event in events.iterrows():

    site = event["site_code"]
    date = event["event_time"]
    region = site.split("-")[0]

    debut = date - pd.DateOffset(months=3)
    fin = date + pd.DateOffset(months=3)

    # Sites témoins : même région, sans événement identifié
    temoins = bats[
        (bats["site_code"].str.startswith(region + "-"))
        & (~bats["site_code"].isin(sites_suspects))
        & (bats["observation_hours"] > 0)
    ].copy()

    avant = temoins[
        (temoins["survey_start"] >= debut)
        & (temoins["survey_start"] < date)
    ]

    apres = temoins[
        (temoins["survey_start"] > date)
        & (temoins["survey_start"] <= fin)
    ]

    if avant["site_code"].nunique() < 2 or apres["site_code"].nunique() < 2:
        continue

    taux_avant = taux_observation(avant)
    taux_apres = taux_observation(apres)

    variation = (
        (taux_apres - taux_avant) / taux_avant * 100
        if taux_avant > 0 else np.nan
    )

    controles.append({
        "site_code": site,
        "event_date": date,
        "variation_temoins": variation,
        "nb_sites_temoins": len(
            set(avant["site_code"]) & set(apres["site_code"])
        )
    })

df_controles = pd.DataFrame(controles)

comparaison_finale = comparison.merge(
    df_controles,
    on=["site_code"],
    how="left"
) if False else comparison.merge(
    df_controles,
    left_on=["site_code", "event_date"],
    right_on=["site_code", "event_date"],
    how="left"
)

comparaison_finale["ecart"] = (
    comparaison_finale["change_pct"]
    - comparaison_finale["variation_temoins"]
)

display(
    comparaison_finale[
        ["site_code", "event_date", "change_pct",
         "variation_temoins", "ecart", "nb_sites_temoins"]
    ].round(2)
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_28676\3979794003.py:78: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2)


,site_code,event_date,change_pct,variation_temoins,ecart,nb_sites_temoins
0,TMB-NAC,2022-02-18 03:00:00,-76.69,-3.53,-73.16,18
1,AVL-MAR,2021-05-21 03:00:00,-70.52,-8.24,-62.28,18
2,KNT-LUM,2019-09-13 03:00:00,-68.95,-16.59,-52.37,19
3,KNT-KES,2021-03-26 03:00:00,-67.93,21.69,-89.62,19
4,TMB-PEL,2024-08-16 03:00:00,-67.89,2.30,-70.18,18
5,AVL-VEL,2023-11-10 03:00:00,-58.23,-7.15,-51.09,18
6,AVL-VEL,2019-11-08 03:00:00,-48.49,2.04,-50.53,18
7,AVL-ORR,2020-06-12 03:00:00,-45.79,-16.11,-29.68,18
8,KNT-KES,2024-03-22 03:00:00,-44.80,21.17,-65.97,19
9,TMB-PEL,2019-08-09 03:00:00,-43.84,-1.36,-42.48,18


les résultats montrent que dans les 15 cas analysés, la diminution du taux d'observation des chauves-souris est plus importante sur les sites concernés que sur les sites témoins.

Plusieurs exemples sont particulièrement révélateurs :

KNT-KES (2021) : baisse de 67,93 % sur le site, contre une augmentation de 21,69 % sur les témoins, soit un écart de −89,62 points.

KNT-KES (2024) : baisse de 44,80 % sur le site, contre une augmentation de 21,17 % sur les témoins, soit un écart de −65,97 points.

TMB-NAC (2022) : baisse de 76,69 % sur le site, contre seulement 3,53 % sur les témoins.

TMB-PEL (2024) : baisse de 67,89 % sur le site, alors que les témoins présentent une légère augmentation de 2,30 %.

AVL-MAR (2021) : baisse de 70,52 % sur le site, contre 8,24 % sur les témoins.

# Lien entre health et logitics

### Examiner les expéditions et les procédures cliniques

In [5]:
# DESTINATIONS DES EXPÉDITIONS BIOLOGIQUES SUSPECTES

shipments = D["shipments"]
facilities = D["erp_facilities"]

# Récupérer les expéditions liées aux événements écologiques
suspect_ids = events["shipment_number"].dropna().unique()

suspect_shipments = shipments[
    shipments["shipment_number"].isin(suspect_ids)
].copy()

# Associer chaque expédition à son établissement de destination
destinations = suspect_shipments.merge(
    facilities[["facility_code", "facility_name"]],
    left_on="destination_facility_code",
    right_on="facility_code",
    how="left",
    validate="many_to_one"
)

# Résumé
print("=== EXPÉDITIONS BIOLOGIQUES SUSPECTES ===")
print("Expéditions distinctes :", suspect_shipments["shipment_number"].nunique())
print("Établissements destinataires :", destinations["destination_facility_code"].nunique())

print("\n=== DESTINATIONS ===")

display(
    destinations.groupby(
        ["destination_facility_code", "facility_name"],
        dropna=False
    ).agg(
        nb_expeditions=("shipment_number", "nunique")
    ).reset_index().sort_values(
        "nb_expeditions", ascending=False
    )
)

print("\n=== PARTENAIRES IMPLIQUÉS ===")

display(
    suspect_shipments.groupby(
        ["sender_partner_code", "receiver_partner_code"]
    ).agg(
        nb_expeditions=("shipment_number", "nunique")
    ).reset_index()
)

=== EXPÉDITIONS BIOLOGIQUES SUSPECTES ===
Expéditions distinctes : 18
Établissements destinataires : 3

=== DESTINATIONS ===


,destination_facility_code,facility_name,nb_expeditions
0,FCL-1019,Meridian Avelora Transfer Annex,7
2,FCL-1024,Meridian Kintara Biological Aggregation Facility,7
1,FCL-1020,Meridian Tembara Coastal Hub,4



=== PARTENAIRES IMPLIQUÉS ===


,sender_partner_code,receiver_partner_code,nb_expeditions
0,BP-30056,BP-30078,18


Les 18 expéditions biologiques suspectes sont envoyées par **Horizon Field Research Cooperative** à **Meridian BioLogistics**.

Elles sont principalement réparties entre trois installations de Meridian :
- **FCL-1024** : 7 expéditions.
- **FCL-1019** : 7 expéditions.
- **FCL-1020** : 4 expéditions.

Cette concentration des expéditions vers un même partenaire et trois installations spécifiques suggère un circuit logistique organisé, qui mérite d'être comparé aux activités médicales identifiées dans les autres fichiers.

In [6]:
procedures = D["clinical_procedures"].copy()

print("Nombre total de procédures :", len(procedures))

# Identifier les codes de procédures les plus fréquents
resume = (
    procedures.groupby("procedure_code", dropna=False)
    .agg(
        nb_procedures=("procedure_code", "size"),
        nb_patients=("patient_number", "nunique"),
        nb_etablissements=("performing_facility_license", "nunique")
    )
    .reset_index()
    .sort_values("nb_procedures", ascending=False)
)

display(resume)

Nombre total de procédures : 310


,procedure_code,nb_procedures,nb_patients,nb_etablissements
6,OCT-TRK,51,48,3
4,MRP-42,50,28,3
5,OCC-PNL,49,47,3
1,AUT-REV,43,41,3
2,CARD-IMG,41,36,3
0,AGE-PNL,39,36,3
3,MOB-PNL,37,34,3


### Observation

Parmi les 310 procédures cliniques, **MRP-42** est le deuxième code le plus fréquent, avec 50 procédures concernant 28 patients. Cela indique que certains patients ont subi plusieurs interventions de ce type.

Nous allons donc nous intéresser à ces procédures et vérifier si elles présentent des correspondances avec les expéditions cliniques de Meridian, notamment au niveau des dates et des établissements.

In [8]:
# Examiner les procédures MRP-42

mrp = procedures[
    procedures["procedure_code"].astype("string")
    .str.strip().str.upper() == "MRP-42"
].copy()

print("PROCÉDURES MRP-42")
print("Nombre de procédures :", len(mrp))

display(
    mrp[
        [
            "procedure_record_id",
            "patient_number",
            "performed_time",
            "performing_facility_license",
            "authorization_class",
            "material_reference",
            "outcome_code"
        ]
    ].head(10)
)

print("\nRÉFÉRENCES DE MATÉRIEL")

display(
    mrp["material_reference"].value_counts(dropna=False)
)

PROCÉDURES MRP-42
Nombre de procédures : 50


,procedure_record_id,patient_number,performed_time,performing_facility_license,authorization_class,material_reference,outcome_code
0,PRC-000001,PAT-31221,2016-02-18 09:05:00,FAC-VLN-101,restricted_specialist,MTL-2016-0002,completed_without_immediate_complication
1,PRC-000002,PAT-31221,2018-02-22 09:05:00,FAC-VLN-101,restricted_specialist,MTL-2018-0003,completed_without_immediate_complication
2,PRC-000003,PAT-31221,2020-03-05 09:05:00,FAC-VLN-101,restricted_specialist,MTL-2020-0004,completed_without_immediate_complication
3,PRC-000004,PAT-31221,2022-02-17 09:05:00,FAC-VLN-101,restricted_specialist,MTL-2022-0005,completed_without_immediate_complication
4,PRC-000005,PAT-31221,2024-03-01 09:05:00,FAC-VLN-101,restricted_specialist,MTL-2024-0006,completed_without_immediate_complication
5,PRC-000006,PAT-31044,2017-05-12 09:05:00,FAC-AVL-201,restricted_specialist,MTL-2017-0007,completed_without_immediate_complication
6,PRC-000007,PAT-31044,2019-05-17 09:05:00,FAC-AVL-201,restricted_specialist,MTL-2019-0008,completed_without_immediate_complication
7,PRC-000008,PAT-31044,2021-06-04 09:05:00,FAC-AVL-201,restricted_specialist,MTL-2021-0009,completed_without_immediate_complication
8,PRC-000009,PAT-31044,2024-08-23 09:05:00,FAC-AVL-201,restricted_specialist,MTL-2024-0010,completed_without_immediate_complication
9,PRC-000010,PAT-31174,2015-09-10 09:05:00,FAC-AVL-201,restricted_specialist,MTL-2015-0011,completed_without_immediate_complication



RÉFÉRENCES DE MATÉRIEL


material_reference
MTL-2016-0002      1
MTL-2018-0003      1
MTL-2020-0004      1
MTL-2022-0005      1
MTL-2024-0006      1
MTL-2017-0007      1
MTL-2019-0008      1
MTL-2021-0009      1
MTL-2024-0010      1
MTL-2015-0011      1
MTL-2017-0012      1
MTL-2019-0013      1
MTL-2021-0014      1
MTL-2018-0015      1
MTL-2020-0016      1
MTL-2022-0017      1
MTL-2024-0018      1
MTL-2017-0019      1
MTL-2019-0020      1
MTL-2022-0021      1
MTL-2025-0022      1
MTL-2016-0023      1
MTL-2018-0024      1
MTL-2020-0025      1
MTL-2019-0026      1
MTL-2021-0027      1
MTL-2023-0028      1
MTL-2014-0029      1
MTL-2016-0030      1
MTL-2018-0031      1
MTL-R-2023-0032    1
MTL-R-2023-0033    1
MTL-R-2023-0034    1
MTL-R-2024-0035    1
MTL-R-2023-0036    1
MTL-R-2021-0037    1
MTL-R-2024-0038    1
MTL-R-2023-0039    1
MTL-R-2021-0040    1
MTL-R-2023-0041    1
MTL-R-2020-0042    1
MTL-R-2023-0043    1
MTL-R-2022-0044    1
MTL-R-2020-0045    1
MTL-R-2022-0046    1
MTL-R-2021-0047    1
MTL-R-2020-0048

Observation

Parmi les 310 procédures cliniques, MRP-42 est le deuxième code le plus fréquent, avec 50 procédures concernant 28 patients. Cela indique que certains patients ont subi plusieurs interventions de ce type.

Nous allons donc nous intéresser à ces procédures et vérifier si elles présentent des correspondances avec les expéditions cliniques de Meridian, notamment au niveau des dates et des établissements.

## identifier les partenaires et les établissements

In [9]:
partners = D["erp_business_partners"]
facilities = D["erp_facilities"]
licensed = D["licensed_facilities"]

# Partenaires des expéditions suspectes
print("=== PARTENAIRES IMPLIQUÉS ===")

display(
    partners.loc[
        partners["partner_code"].isin(["BP-30056", "BP-30078"]),
        ["partner_code", "partner_name"]
    ]
)

# Installations logistiques identifiées
print("\n=== INSTALLATIONS LOGISTIQUES ===")

display(
    facilities.loc[
        facilities["facility_code"].isin(
            ["FCL-1019", "FCL-1020", "FCL-1024", "FCL-1030"]
        ),
        [
            "facility_code",
            "facility_name",
            "operating_partner_code",
            "region"
        ]
    ]
)

# Établissements médicaux d'Asterion
print("\n=== ÉTABLISSEMENTS MÉDICAUX D'ASTERION ===")

display(
    licensed.loc[
        licensed["facility_license_number"].isin(
            ["FAC-VLN-101", "FAC-AVL-201"]
        ),
        [
            "facility_license_number",
            "facility_name",
            "region",
            "declared_services"
        ]
    ]
)

=== PARTENAIRES IMPLIQUÉS ===


,partner_code,partner_name
55,BP-30056,Horizon Field Research Cooperative
77,BP-30078,Meridian BioLogistics



=== INSTALLATIONS LOGISTIQUES ===


,facility_code,facility_name,operating_partner_code,region
1,FCL-1024,Meridian Kintara Biological Aggregation Facility,BP-30078,Kintara Karst Cluster
2,FCL-1019,Meridian Avelora Transfer Annex,BP-30078,Eastern Avelora Agricultural Cluster
3,FCL-1020,Meridian Tembara Coastal Hub,BP-30078,Tembara Coastal Cluster
7,FCL-1030,Kintara Cave Authority Field Depot,BP-30016,Kintara Karst Cluster



=== ÉTABLISSEMENTS MÉDICAUX D'ASTERION ===


,facility_license_number,facility_name,region,declared_services
0,FAC-VLN-101,Asterion Central Clinical Campus,Valenne Republic,preventive medicine; cardiac assessment; clinical research
1,FAC-AVL-201,Asterion Avelora Executive Clinic,Avelora Federation,executive health; specialist follow-up; outpatient procedures


Les expéditions biologiques suspectes impliquent deux organisations : **Horizon Field Research Cooperative** et **Meridian BioLogistics**.

Plusieurs installations logistiques sont associées à Meridian. Par ailleurs, les établissements médicaux d'**Asterion Clinical Institute** sont identifiés dans les registres publics.

Ces informations permettent d'identifier les acteurs et les établissements à rapprocher dans la suite de l'analyse, sans démontrer à ce stade de transfert entre les installations logistiques et les établissements médicaux.

## Expéditions impliquant Horizon ou Meridian


In [10]:
# Expéditions impliquant Horizon ou Meridian
related_shipments = shipments[
    shipments["sender_partner_code"].isin(["BP-30056", "BP-30078"])
    | shipments["receiver_partner_code"].isin(["BP-30056", "BP-30078"])
].copy()

print("=== TYPES DE LIVRAISONS ===")

display(
    related_shipments.groupby(
        ["sender_partner_code", "receiver_partner_code", "declared_purpose"],
        dropna=False
    ).size().reset_index(name="nb_expeditions")
    .sort_values("nb_expeditions", ascending=False)
)

# Livraisons cliniques envoyées par Meridian
clinical_shipments = shipments[
    (shipments["sender_partner_code"] == "BP-30078")
    & (shipments["declared_purpose"].astype("string")
       .str.contains("clinical", case=False, na=False))
].copy()

print("\n=== LIVRAISONS CLINIQUES DE MERIDIAN ===")
print("Nombre d'expéditions :", len(clinical_shipments))

display(
    clinical_shipments[[
        "shipment_number",
        "arrival_time",
        "receiver_partner_code",
        "destination_facility_code",
        "declared_purpose"
    ]].head(10)
)

=== TYPES DE LIVRAISONS ===


,sender_partner_code,receiver_partner_code,declared_purpose,nb_expeditions
47,BP-30078,BP-30021,temperature-controlled clinical programme delivery,26
24,BP-30056,BP-30078,environmental and wildlife reference materials,18
46,BP-30078,BP-30021,scheduled stock transfer,8
41,BP-30078,BP-30021,agricultural distribution,6
45,BP-30078,BP-30021,laboratory delivery,6
...,...,...,...,...
64,BP-30078,BP-30065,facility replenishment,1
65,BP-30078,BP-30071,customer distribution,1
66,BP-30078,BP-30076,field programme support,1
67,BP-30078,BP-30079,laboratory delivery,1



=== LIVRAISONS CLINIQUES DE MERIDIAN ===
Nombre d'expéditions : 26


,shipment_number,arrival_time,receiver_partner_code,destination_facility_code,declared_purpose
0,SHP-2014-00001,2014-07-11T03:00:00,BP-30021,FCL-1007,temperature-controlled clinical programme delivery
11,SHP-2015-00011,2015-09-04T05:00:00,BP-30021,FCL-1007,temperature-controlled clinical programme delivery
19,SHP-2016-00002,2016-02-11T22:00:00,BP-30021,FCL-1010,temperature-controlled clinical programme delivery
23,SHP-2016-00006,2016-06-11T11:00:00,BP-30021,FCL-1010,temperature-controlled clinical programme delivery
24,SHP-2016-00007,2016-07-22T22:00:00,BP-30021,FCL-1007,temperature-controlled clinical programme delivery
40,SHP-2017-00006,2017-05-06T05:00:00,BP-30021,FCL-1007,temperature-controlled clinical programme delivery
49,SHP-2017-00015,2017-09-15T23:00:00,BP-30021,FCL-1007,temperature-controlled clinical programme delivery
51,SHP-2017-00017,2017-10-26T15:00:00,BP-30021,FCL-1010,temperature-controlled clinical programme delivery
54,SHP-2018-00001,2018-01-19T22:00:00,BP-30021,FCL-1010,temperature-controlled clinical programme delivery
56,SHP-2018-00003,16/02/2018 12:00:00,BP-30021,FCL-1010,temperature-controlled clinical programme delivery


L'analyse met en évidence deux circuits logistiques impliquant **Meridian BioLogistics** :

- Des expéditions d'échantillons biologiques provenant de **Horizon Field Research Cooperative**.
- **26 livraisons cliniques** envoyées par Meridian à **Asterion Clinical Institute**.

Meridian apparaît ainsi comme un intermédiaire commun entre les activités écologiques et médicales.

Cette correspondance entre organisations constitue un indice intéressant, mais ne prouve pas que les échantillons biologiques ont été utilisés dans les établissements cliniques.

## correspondances entre les livraisons cliniques et les procédures MRP-42

In [11]:
# Préparer les dates
clinical = clinical_shipments.copy()
clinical["arrival_time"] = pd.to_datetime(
    clinical["arrival_time"], format="mixed", errors="coerce"
)

mrp_analyse = mrp.copy()
mrp_analyse["performed_time"] = pd.to_datetime(
    mrp_analyse["performed_time"], format="mixed", errors="coerce"
)

# Correspondance entre codes logistiques et licences médicales
facility_mapping = {
    "FCL-1010": "FAC-VLN-101",
    "FCL-1007": "FAC-AVL-201"
}

clinical["facility_license"] = (
    clinical["destination_facility_code"].map(facility_mapping)
)

# Jointure sur le même établissement
correspondances = clinical.merge(
    mrp_analyse,
    left_on="facility_license",
    right_on="performing_facility_license",
    how="inner",
    suffixes=("_shipment", "_procedure")
)

# Délai entre livraison et procédure
correspondances["delai_jours"] = (
    correspondances["performed_time"]
    - correspondances["arrival_time"]
).dt.total_seconds() / 86400

# Procédures dans les 30 jours suivant la livraison
correspondances = correspondances[
    correspondances["delai_jours"].between(0, 30)
].copy()

# Résumé
print("=== CORRESPONDANCES LIVRAISONS / PROCÉDURES MRP-42 ===")
print("Correspondances :", len(correspondances))
print("Expéditions distinctes :", correspondances["shipment_number"].nunique())
print("Procédures distinctes :", correspondances["procedure_record_id"].nunique())

display(
    correspondances[[
        "shipment_number",
        "destination_facility_code",
        "procedure_record_id",
        "performing_facility_license",
        "delai_jours"
    ]].sort_values("delai_jours").head(10).round(2)
)

=== CORRESPONDANCES LIVRAISONS / PROCÉDURES MRP-42 ===
Correspondances : 30
Expéditions distinctes : 26
Procédures distinctes : 28


,shipment_number,destination_facility_code,procedure_record_id,performing_facility_license,delai_jours
259,SHP-2018-00034,FCL-1010,PRC-000023,FAC-VLN-101,4.17
468,SHP-2021-00039,FCL-1007,PRC-000013,FAC-AVL-201,4.25
488,SHP-2022-00001,FCL-1010,PRC-000016,FAC-VLN-101,4.55
369,SHP-2020-00004,FCL-1010,PRC-000015,FAC-VLN-101,4.71
82,SHP-2016-00006,FCL-1010,PRC-000022,FAC-VLN-101,4.92
320,SHP-2019-00038,FCL-1007,PRC-000012,FAC-AVL-201,5.30
143,SHP-2017-00015,FCL-1007,PRC-000011,FAC-AVL-201,5.42
578,SHP-2023-00003,FCL-1007,PRC-000027,FAC-AVL-201,5.42
621,SHP-2024-00005,FCL-1010,PRC-000005,FAC-VLN-101,5.71
217,SHP-2018-00003,FCL-1010,PRC-000002,FAC-VLN-101,5.88


Le rapprochement des livraisons cliniques de Meridian avec les procédures MRP-42, en tenant compte des établissements et d'un délai maximal de 30 jours, met en évidence **30 correspondances**, impliquant **26 expéditions distinctes et 28 procédures distinctes**.

Ces résultats montrent une proximité géographique et temporelle entre les livraisons et les interventions médicales. Ils suggèrent un lien potentiel entre les deux activités, sans prouver que les produits livrés ont été utilisés lors des procédures.

# Lien entre health, logistics et public records

## Documents reliant les expéditions au programme CHAMELEO


In [12]:
# Documents reliant les expéditions au programme CHAMELEO
documents = D["public_and_investigative_documents"]
memos = D["institutional_memos"]
officers = D["officer_filings"]

print("=== TÉMOIGNAGE SUR LES ÉCHANTILLONS BIOLOGIQUES ===")
display(
    documents.loc[
        documents["document_id"] == "PUB-510079",
        ["document_id", "title", "text"]
    ]
)

print("\n=== GOUVERNANCE DU PROGRAMME CHAMELEO ===")
display(
    memos.loc[
        memos["document_id"] == "MEM-510101",
        ["document_id", "subject_line", "text"]
    ]
)

print("\n=== DIRECTION DE MERIDIAN ===")
display(
    officers.loc[
        officers["full_name"].str.contains("Elias Voss", case=False, na=False),
        ["full_name", "role_title", "registration_number"]
    ]
)

=== TÉMOIGNAGE SUR LES ÉCHANTILLONS BIOLOGIQUES ===


,document_id,title,text
5,PUB-510079,Message Concerning Collected Eonycteris,I worked on restricted receiving and reconciliation records. I am sending this because the standard files make the programme l...



=== GOUVERNANCE DU PROGRAMME CHAMELEO ===


,document_id,subject_line,text
2,MEM-510101,Restricted CHAMELEO Governance Review,"MEETING RECORD — RESTRICTED GOVERNANCE REVIEW\r\n\r\nAttendees included the executive chair, the secure-delivery manager, clin..."



=== DIRECTION DE MERIDIAN ===


,full_name,role_title,registration_number
3,Dr. Elias Voss,Executive chair of Meridian BioLogistics,REG-VLN-01001


Lien entre les expéditions biologiques et CHAMELEO

Le document PUB-510079 apporte un nouvel indice sur les expéditions biologiques suspectes : un témoignage indique que des chauves-souris Eonycteris spelaea étaient collectées, transportées dans des unités scellées, puis retirées du suivi ordinaire des stocks. Il mentionne également le code CHAMELEO.

Le mémo MEM-510101 révèle que CHAMELEO est associé à un système d'autorisations confidentielles impliquant la direction de Meridian BioLogistics. Il introduit aussi la désignation Unwabu, liée à l'autorité d'approbation du programme.

Enfin, les registres publics identifient Dr Elias Voss comme dirigeant de Meridian.

Ces éléments renforcent l'hypothèse d'un programme organisé autour des échantillons biologiques, dont la gestion implique Meridian. Toutefois, la destination finale des échantillons et leur éventuelle utilisation médicale restent à démontrer.

# Analyse des patients — Recherche de profils atypiques

In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Charger les observations médicales
obs = D["clinical_observations"].copy()

# Une ligne par patient, avec la moyenne de chaque mesure
feat = obs.pivot_table(
    index="patient_number",
    columns="observation_code",
    values="numeric_value",
    aggfunc="mean"
)

# Standardisation et clustering en 3 groupes
X = StandardScaler().fit_transform(
    feat.fillna(feat.median())
)

feat["cluster"] = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=0
).fit_predict(X)

# Identifier le plus petit groupe
effectifs = feat["cluster"].value_counts()
small = effectifs.idxmin()
group = feat[feat["cluster"] == small].index

print("=== GROUPES DE PATIENTS ===")
display(effectifs.rename_axis("cluster").to_frame("nb_patients"))

print("Patients du groupe minoritaire :", len(group))

=== GROUPES DE PATIENTS ===


,nb_patients
cluster,
0,114
2,111
1,15


Patients du groupe minoritaire : 15


Le clustering K-means répartit les 240 patients en trois groupes : deux groupes majoritaires de 114 et 111 patients, et un groupe minoritaire de 15 patients (6,25 %).

Ce dernier présente un profil de mesures médicales suffisamment différent pour être distingué par l'algorithme. Il constitue donc une piste intéressante pour rechercher d'éventuelles particularités cliniques.

Cependant, l'appartenance à ce groupe ne signifie pas nécessairement que ces patients présentent des anomalies médicales.

In [14]:
from collections import Counter

notes = D["clinical_notes"].copy()

# Identifier les notes contenant des phrases inhabituelles
sentences = notes["text"].fillna("").str.split(r"(?<=\.)\s+")

counts = Counter(
    s.strip()
    for doc in sentences
    for s in doc
    if s.strip()
)

notes["unique_share"] = sentences.apply(
    lambda doc: np.mean([
        counts[s.strip()] == 1
        for s in doc if s.strip()
    ]) if any(s.strip() for s in doc) else 0
)

# Notes dont plus de 50 % des phrases sont uniques
special = notes[notes["unique_share"] > 0.5].copy()

# Vérifier leur présence dans le groupe minoritaire
special["groupe_atypique"] = special["patient_number"].isin(group)

print("=== NOTES CLINIQUES INHABITUELLES ===")
print("Notes inhabituelles :", len(special))
print("Notes concernant le groupe atypique :",
      special["groupe_atypique"].sum())

display(
    special[[
        "document_id",
        "patient_number",
        "title",
        "groupe_atypique"
    ]].head(10)
)

=== NOTES CLINIQUES INHABITUELLES ===
Notes inhabituelles : 4
Notes concernant le groupe atypique : 4


,document_id,patient_number,title,groupe_atypique
0,CLN-510027,PAT-31065,Targeted Thoracic Imaging Review,True
1,CLN-510055,PAT-31115,Consultation on Recurrent Capture Discrepancies,True
2,CLN-510025,PAT-31065,Restricted Phenotype and Public-Identity Resolution,True
3,CLN-510064,PAT-31012,Persistent Multisystem Follow-up,True


L'analyse identifie 4 notes cliniques inhabituelles, qui concernent toutes des patients appartenant au cluster minoritaire de 15 patients.

Ces notes sont associées à 3 patients distincts. Leurs titres évoquent notamment des examens thoraciques, des divergences récurrentes dans les enregistrements, des questions d'identité et un suivi médical multisystémique.

Cette correspondance renforce l'intérêt du groupe minoritaire identifié par K-means. Elle suggère que certains patients présentant des profils de mesures différents font également l'objet de comptes rendus cliniques particuliers.

Toutefois, le caractère inhabituel des notes repose ici sur leur singularité textuelle et ne constitue pas, à lui seul, une preuve d'anomalie médicale.

In [17]:
docs = pd.concat([
    D["public_and_investigative_documents"].assign(source="public"),
    D["institutional_memos"].assign(source="memo")
], ignore_index=True)

texte = docs["text"].fillna("")

mask = (
    texte.str.contains("Meridian", case=False)
    & texte.str.contains("Asterion|CHAMELEO|biological", case=False, regex=True)
)

display(
    docs.loc[mask, [
        "source", "text"
    ]]
)

,source,text
42,memo,"MEETING RECORD — RESTRICTED GOVERNANCE REVIEW\r\n\r\nAttendees included the executive chair, the secure-delivery manager, clin..."


In [ ]:
print(docs.loc[mask, "text"].iloc[0])

MEETING RECORD — RESTRICTED GOVERNANCE REVIEW

Attendees included the executive chair, the secure-delivery manager, clinical programme counsel, and the records secretary.

AGENDA
1. Authority for CHAMELEO routing.
2. Relationship between programme designation and executive approvals.
3. Separation of ordinary corporate records from restricted appendices.

DISCUSSION
The chair confirmed that Unwabu is the designation used in the restricted governance schedule for the authority approving CHAMELEO exceptions. The same schedule assigns final escalation to the Meridian executive office. Participants agreed that ordinary employees should see only shipment, inventory, and compliance fields needed for their roles. The secretary noted that earlier instructions treated CHAMELEO as a handling code, while the governance schedule links it to a named authority.

DECISIONS
The secure-delivery manager will maintain the restricted appendix. Compliance will retain a cross-reference between the designati

In [20]:
enc = D["clinical_encounters"]

print("Colonnes disponibles :", enc.columns.tolist())

display(
    enc.loc[
        enc["patient_number"] == "PAT-31065"
    ]
)

Colonnes disponibles : ['encounter_number', 'patient_number', 'encounter_start', 'encounter_end', 'facility_license_number', 'encounter_type', 'referral_source', 'attending_license_number', 'service_unit', 'disposition', 'confidentiality_class', 'follow_up_date']


,encounter_number,patient_number,encounter_start,encounter_end,facility_license_number,encounter_type,referral_source,attending_license_number,service_unit,disposition,confidentiality_class,follow_up_date
33,ENC-000034,PAT-31065,2018-09-06 09:15:00,2018-09-06 10:59:00,FAC-VLN-101,cardiac_assessment,research programme,LIC-VLN-MED-1006,specialist review,follow_up_planned,confidential,2020-02-28
34,ENC-000035,PAT-31065,2020-02-28 10:15:00,2020-02-28 11:11:00,FAC-VLN-101,cardiac_assessment,executive health office,LIC-VLN-MED-1006,cardiac diagnostics,follow_up_planned,standard,2021-07-18
35,ENC-000036,PAT-31065,2021-07-18 11:15:00,2021-07-18 12:32:00,FAC-VLN-101,cardiac_assessment,research programme,LIC-VLN-MED-1003,general assessment,follow_up_planned,confidential,2023-01-19
36,ENC-000037,PAT-31065,2023-01-19 12:15:00,2023-01-19 13:59:00,FAC-VLN-101,mobility_assessment,self,LIC-VLN-MED-1006,mobility laboratory,follow_up_planned,standard,2024-06-12
37,ENC-000038,PAT-31065,2024-06-12 13:15:00,2024-06-12 15:09:00,FAC-VLN-101,specialist_follow_up,primary care,LIC-VLN-MED-1002,cardiac diagnostics,follow_up_planned,confidential,2026-01-11
38,ENC-000039,PAT-31065,2026-01-11 14:15:00,2026-01-11 15:47:00,FAC-VLN-101,ocular_assessment,primary care,LIC-VLN-RES-1016,general assessment,discharged,confidential,NaN


## Conclusion — Synthèse des découvertes et hypothèses

### 1. Collecte suspecte de chauves-souris
- Dans les données écologiques, nous avons observé une diminution importante des chauves-souris, particulièrement de l'espèce ***Eonycteris spelaea***, sur plusieurs sites après des interventions.
- Ces interventions sont associées à des expéditions d'échantillons biologiques de **Horizon Field Research Cooperative vers Meridian BioLogistics**.
- Un témoignage retrouvé dans les documents publics affirme que les cargaisons contiennent des chauves-souris collectées, vivantes ou récemment capturées, destinées à être « consommées ».
- **Notre hypothèse :** les chauves-souris seraient collectées dans le cadre d'un programme organisé, dont l'objectif réel n'est pas clairement déclaré.

### 2. Un circuit logistique reliant Horizon, Meridian et Asterion
- Nous avons identifié 18 expéditions biologiques associées à des ajustements négatifs de stock chez Meridian.
- Meridian effectue également **26 livraisons cliniques vers Asterion Clinical Institute**, dont certaines précèdent de quelques jours des procédures médicales MRP-42.
- Des réceptions d'échantillons biologiques provenant de Meridian ont également été retrouvées dans les établissements d'Asterion.
- **Notre hypothèse :** Meridian pourrait servir d'intermédiaire entre la collecte des chauves-souris et certaines activités médicales d'Asterion. Nous n'avons cependant pas démontré que les échantillons livrés sont ceux provenant de Horizon.

### 3. Des patients présentant des caractéristiques médicales atypiques
- L'analyse de 240 patients par clustering a permis d'identifier un groupe minoritaire de **15 patients atypiques**.
- Plusieurs notes médicales décrivent des particularités cardiaques, motrices et oculaires inhabituelles.
- Ces patients sont suivis dans différents établissements, notamment ceux d'Asterion.
- **Notre hypothèse :** ces patients pourraient être liés à un programme médical particulier. Nous cherchons notamment à déterminer si leurs anomalies ont un rapport avec les activités biologiques identifiées.

### 4. Elias Voss et une possible identité traversant les siècles
- Les registres publics indiquent qu'**Elias Voss est le dirigeant de Meridian BioLogistics**, l'entreprise impliquée dans les expéditions.
- Une note clinique établit une correspondance fortement étayée entre Elias Voss et le patient atypique `PAT-31065`, qui présente notamment des anomalies cardiaques, motrices et oculaires.
- L'analyse des documents historiques et des registres publics fait également apparaître une **possible continuité d'identité à travers plusieurs siècles**, avec une succession de noms : **Warden → Vosmere → … → Voss**.
- **Notre hypothèse :** ces différentes identités pourraient correspondre à une même personne qui aurait traversé les siècles en changeant de nom. Elias Voss pourrait donc être directement concerné par le secret, au-delà de son rôle de dirigeant. Cette continuité historique reste toutefois à confirmer.

### 5. Un programme confidentiel : CHAMELEO
- Plusieurs documents mentionnent **CHAMELEO**, un programme impliquant des procédures logistiques restreintes et des autorisations particulières.
- Le nom **Unwabu** apparaît comme une désignation d'autorité associée aux validations de ce programme et à la direction de Meridian.
- Les témoignages évoquent une collecte nocturne de chauves-souris, leur acheminement par le réseau logistique de Meridian et une destination finale qui n'est pas clairement identifiée.
- **Notre hypothèse :** CHAMELEO pourrait coordonner certaines opérations confidentielles de collecte et de circulation de matériel biologique, potentiellement liées aux autres anomalies découvertes.

### Hypothèse générale sur le secret

Un réseau dirigé ou coordonné par Elias Voss et Meridian BioLogistics organiserait la collecte clandestine de chauves-souris, notamment de l'espèce Eonycteris spelaea, dissimulée derrière des activités de recherche scientifique et des transports biologiques réglementés. Les échantillons seraient acheminés depuis Horizon vers Meridian, puis potentiellement destinés à une consommation ou à une utilisation biologique particulière, comme le suggère un témoignage retrouvé dans les documents publics.

Ces ressources pourraient être utilisées dans le cadre d'un programme confidentiel, CHAMELEO, potentiellement lié aux activités médicales d'Asterion et à des individus présentant des caractéristiques physiologiques inhabituelles.

Par ailleurs, les documents historiques suggèrent qu'une même personne pourrait avoir traversé plusieurs siècles sous différentes identités (Warden → Vosmere → … → Voss). Ces changements de nom pourraient servir à dissimuler une longévité exceptionnelle.

Cette hypothèse est renforcée par le rapprochement entre Elias Voss et le patient atypique PAT-31065, qui présente notamment des anomalies cardiaques, motrices et oculaires.
